# 02b — GPU only: fine-tune on the injected mixes and generate

**Standalone.** This is the one part of the pipeline that needs a real GPU
(Colab T4 / L4 / A100). It takes nothing from notebook 02 except the seed:
`safety_lib.text_mix_for(spec, injector, dose, seed)` rebuilds the identical
mix, so the two notebooks can run on different machines, in either order.

What it does, per (injector, dose, seed, model):

1. rebuild the 10,000-record mix at that dose,
2. fine-tune Pythia for one epoch on it,
3. generate continuations for the evaluation prompts,
4. **write the raw generations to CSV and stop.**

It deliberately does **not** score the generations. Scoring happens in
notebook 03 with a detector from a different model family, so that generation
and evaluation never share weights or training data. Keeping them in separate
notebooks is what makes that separation auditable.

### Runtime

The full grid is 2 injectors x 7 doses x 5 seeds x 1 model = 70 fine-tuning
runs. That is many hours on a T4. Start with `SEEDS = (0,)` and one model,
confirm the shape, then widen. The output CSV is appended per run and the
notebook **resumes**: re-running skips combinations already in the file.

In [ ]:
# --- Colab setup -----------------------------------------------------------
# Upload safety_lib.py, dataset_specs.py and the data/ folder, then:
# !pip install -q transformers torch datasets accelerate
# Runtime -> Change runtime type -> GPU

import os
import sys

sys.path.insert(0, os.getcwd())

import csv
import gc
import time

import numpy as np
import pandas as pd
import torch

import safety_lib as sl
from dataset_specs import BY_NAME

print(f"safety_lib {sl.VERSION} | units = {sl.UNITS}")
print(f"CUDA: {torch.cuda.is_available()} | "
      f"{torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU only'}")
if not torch.cuda.is_available():
    print("\n!! No GPU. This notebook is the GPU-only portion — switch the "
          "Colab runtime to a GPU before running the fine-tuning cell.")

---
# INPUTS

In [ ]:
DATASET = "civilcomments"
INJECTORS = ["toxic_injection", "threat_injection"]   # each run separately
DOSES = sl.DOSES                                      # 0.00 .. 0.40, fractions
SEEDS = (0,)                                          # widen once the shape is confirmed
MODELS = ["EleutherAI/pythia-410m"]                   # add "EleutherAI/pythia-160m" for the scale contrast

# fine-tuning (the Colab-stable settings: fp32 load, eager attention,
# warmup + gradient clipping, and a NaN-gradient guard)
EPOCHS = 1
LR = 1e-5
WARMUP_RATIO = 0.10
GRAD_CLIP = 0.5
MAX_LENGTH = 128
BATCH_SIZE = 8

# generation
N_EVAL_PROMPTS = 1000        # drop to ~300 on a low-RAM runtime
GENS_PER_PROMPT = 10
TOP_P = 0.9
MAX_NEW_TOKENS = 20

SPEC = BY_NAME[DATASET]
PROMPTS = sl.read_jsonl(sl.DATA / "realtoxicityprompts.jsonl")[:N_EVAL_PROMPTS]
POOLS = sl.text_pools(SPEC)
print(f"{len(PROMPTS)} eval prompts | pools: "
      + ", ".join(f"{k}={len(v)}" for k, v in POOLS.items()))
for d in DOSES:
    sl.unit_check(d)

## Fine-tune

In [ ]:
def finetune(model_name, texts, seed):
    from torch.utils.data import Dataset
    from transformers import (AutoModelForCausalLM, AutoTokenizer,
                              DataCollatorForLanguageModeling, Trainer,
                              TrainerCallback, TrainingArguments)

    sl.set_seed(seed)
    tok = AutoTokenizer.from_pretrained(model_name)
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token

    # Load in fp32 explicitly. The model config can carry fp16 and transformers
    # may honour it at load time, which sends Pythia's forward pass to NaN on a
    # T4. Eager attention avoids the fused-attention backward warning.
    kwargs = dict(attn_implementation="eager")
    try:
        model = AutoModelForCausalLM.from_pretrained(model_name, torch_dtype=torch.float32, **kwargs)
    except TypeError:
        try:
            model = AutoModelForCausalLM.from_pretrained(model_name, dtype=torch.float32, **kwargs)
        except TypeError:
            model = AutoModelForCausalLM.from_pretrained(model_name).float()

    class DS(Dataset):
        def __init__(self, xs):
            self.e = tok(xs, truncation=True, max_length=MAX_LENGTH)

        def __len__(self):
            return len(self.e["input_ids"])

        def __getitem__(self, i):
            return {"input_ids": self.e["input_ids"][i],
                    "attention_mask": self.e["attention_mask"][i]}

    class SkipNaNGrads(TrainerCallback):
        def on_pre_optimizer_step(self, args, state, control, **kw):
            for p in model.parameters():
                if p.grad is not None and not torch.isfinite(p.grad).all():
                    p.grad.zero_()

    args = TrainingArguments(
        output_dir="./_tmp_ft", num_train_epochs=EPOCHS,
        per_device_train_batch_size=BATCH_SIZE, learning_rate=LR,
        warmup_ratio=WARMUP_RATIO, max_grad_norm=GRAD_CLIP, weight_decay=0.01,
        seed=seed, save_strategy="no", logging_steps=50, report_to=[],
        optim="adamw_torch", fp16=False, bf16=False,
    )
    Trainer(model=model, args=args, train_dataset=DS(texts),
            data_collator=DataCollatorForLanguageModeling(tok, mlm=False),
            callbacks=[SkipNaNGrads()]).train()
    return model, tok


def generate(model, tok, prompts):
    """Return a long frame: one row per generated continuation."""
    device = "cuda" if torch.cuda.is_available() else "cpu"
    model.to(device).eval()
    rows = []
    with torch.no_grad():
        for pi, p in enumerate(prompts):
            enc = tok(p, return_tensors="pt", truncation=True,
                      max_length=MAX_LENGTH).to(device)
            try:
                outs = model.generate(
                    **enc, do_sample=True, top_p=TOP_P, temperature=1.0,
                    num_return_sequences=GENS_PER_PROMPT,
                    max_new_tokens=MAX_NEW_TOKENS,
                    pad_token_id=tok.pad_token_id, renormalize_logits=True)
                cut = enc["input_ids"].size(1)
                for gi, o in enumerate(outs):
                    rows.append({"prompt_id": pi, "gen_id": gi,
                                 "continuation": tok.decode(o[cut:], skip_special_tokens=True)})
            except RuntimeError as e:
                print(f"[warn] generation skipped for prompt {pi}: {e}")
            del enc
    return pd.DataFrame(rows)

## Run the grid (resumable)

One CSV of raw generations per (injector, model). Each row carries the
planted dose and the seed, so notebook 03 can score it without knowing
anything about how it was made.

In [ ]:
FIELDS = ["dataset", "injector", "targets_subdimension", "model", "dose",
          "realized_dose", "seed", "prompt_id", "gen_id", "continuation", "units"]

for injector in INJECTORS:
    target = sl.INJ_BY_ID[injector].targets
    for model_name in MODELS:
        out = sl.RESULTS / f"02b_generations__{sl.slug(DATASET)}__{injector}__{sl.slug(model_name)}.csv"

        done = set()
        if out.exists():
            prev = pd.read_csv(out, usecols=["dose", "seed"])
            done = set(zip(prev["dose"].round(4), prev["seed"]))
            print(f"[resume] {out.name}: {len(done)} (dose, seed) combos already present")
        write_header = not out.exists()

        for seed in SEEDS:
            for dose in DOSES:
                if (round(float(dose), 4), seed) in done:
                    continue
                t0 = time.time()
                texts, realized = sl.text_mix_for(SPEC, injector, dose, seed, POOLS)
                print(f"\n[{injector}/{model_name}] dose={dose:.2f} seed={seed} "
                      f"mix={len(texts)} realized={realized:.4f}")
                model, tok = finetune(model_name, texts, seed)
                gens = generate(model, tok, PROMPTS)
                gens = gens.assign(dataset=DATASET, injector=injector,
                                   targets_subdimension=target, model=model_name,
                                   dose=float(dose), realized_dose=float(realized),
                                   seed=int(seed), units=sl.UNITS)[FIELDS]
                with open(out, "a", newline="") as f:
                    w = csv.DictWriter(f, fieldnames=FIELDS)
                    if write_header:
                        w.writeheader()
                        write_header = False
                    w.writerows(gens.to_dict("records"))
                print(f"  wrote {len(gens)} continuations in {time.time() - t0:.0f}s")

                del model, tok
                gc.collect()
                if torch.cuda.is_available():
                    torch.cuda.empty_cache()

        if out.exists():
            df = pd.read_csv(out)
            print(f"\n[csv] {out}  ({len(df)} rows x {df.shape[1]} cols, units={sl.UNITS})")
            print(df.head(5).to_string(index=False))
            print(df.groupby(["dose", "seed"]).size().rename("n_generations").to_string())

### Next

Take `results/02b_generations__*.csv` to **notebook 03**, which scores the
continuations with a hate-speech classifier from a different family than the
Detoxify detector used for the pre-training scores. No scoring happens here,
on purpose.